In [0]:
%sql
describe olist.bronze.orders

In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.orders AS
SELECT
    order_id,
    customer_id,
    UPPER(TRIM(order_status)) AS order_status,

    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,

    -- Inconsistencia: enviado antes de ser aprobado
    CASE
        WHEN order_approved_at IS NOT NULL
         AND order_delivered_carrier_date IS NOT NULL
         AND order_delivered_carrier_date < order_approved_at
        THEN TRUE
        ELSE FALSE
    END AS is_shipped_before_approval,

    -- Inconsistencia: entregado al cliente antes de pasar al transportista
    CASE
        WHEN order_delivered_carrier_date IS NOT NULL
         AND order_delivered_customer_date IS NOT NULL
         AND order_delivered_customer_date < order_delivered_carrier_date
        THEN TRUE
        ELSE FALSE
    END AS is_delivered_before_carrier,

    -- Indicador de negocio: pedido entregado tarde
    CASE
        WHEN order_delivered_customer_date IS NOT NULL
         AND order_estimated_delivery_date IS NOT NULL
         AND order_delivered_customer_date > order_estimated_delivery_date
        THEN TRUE
        ELSE FALSE
    END AS is_late_delivery

FROM olist.bronze.orders
WHERE order_id IS NOT NULL;

In [0]:
%sql
select * from olist.silver.orders

# Resumen - Silver Orders

En este notebook se construyó la tabla `olist.silver.orders` a partir de los datos disponibles en `olist.bronze.orders`.

## Transformaciones realizadas

- Se conservaron los identificadores `order_id` y `customer_id`.
- Se estandarizó `order_status` eliminando espacios y convirtiendo los valores a mayúsculas.
- Se conservaron las columnas temporales originales, ya que fueron correctamente inferidas como `TIMESTAMP`.
- Se excluyeron registros con `order_id` nulo.

## Indicadores de calidad

Se añadieron columnas booleanas para identificar inconsistencias temporales:

- `is_shipped_before_approval`: identifica pedidos enviados al transportista antes de ser aprobados.
- `is_delivered_before_carrier`: identifica pedidos entregados al cliente antes de la fecha registrada de entrega al transportista.

Los registros inconsistentes no fueron eliminados, sino conservados y marcados para mantener trazabilidad.

## Indicador de negocio

- `is_late_delivery`: identifica pedidos cuya fecha real de entrega fue posterior a la fecha estimada.

Este indicador será utilizado posteriormente para analizar posibles relaciones entre retrasos y experiencia del cliente.

## Resultado

Se generó una tabla Silver limpia y estandarizada, manteniendo los datos relevantes para análisis posteriores y agregando indicadores de calidad y negocio.